In [1]:
# Install compatible dependencies
!pip install --upgrade numpy pandas scikit-learn peft wandb tqdm -q

In [ ]:
import os
os.kill(os.getpid(), 9)

# SOMA — Experiment 1: Permuted MNIST

**Wakasa Labs · Nairobi, Kenya · March 2026**

This notebook runs the full SOMA system on 10 permuted MNIST tasks.
Designed to run on **Kaggle T4 GPU** (~15 min runtime).

**PASS criterion:** BT > -0.05 AND K < 10

In [2]:
# Clone SOMA repo if not exists and add to path
import os
import sys

if not os.path.exists('soma_research'):
    !git clone https://github.com/LensenWakasa/SOMA-research.git soma_research

sys.path.insert(0, os.path.abspath('soma_research'))

# Verify import
from soma.core.necessity import SomaNecessity
from soma.core.grow import SomaGrow
from soma.core.learn import SomaLearn
print('SOMA loaded successfully')

SOMA loaded successfully


In [5]:
# GPU check
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    # Fixed attribute name from total_mem to total_memory
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [20]:
# 1. Apply Rule-based Warmup and Disable MERGE via source patching
import os
import importlib

# Patch learn.py for Warmup Logic
learn_path = 'soma_research/soma/core/learn.py'
with open(learn_path, 'r') as f: learn_content = f.read()

# Inject warmup logic into the step function
warmup_patch = """
        # --- Rule-based Warmup Patch ---
        is_warmup = task_idx < 4
        if is_warmup:
            action = 1 if nec_result.necessity else 0 # 1=SPAWN, 0=UPDATE
            # Bypass policy selection during warmup
"""
learn_content = learn_content.replace("action, log_prob = self.policy.select_action(state)",
                                     warmup_patch + "        action, log_prob = (action, torch.tensor(0.0)) if is_warmup else self.policy.select_action(state)")
with open(learn_path, 'w') as f: f.write(learn_content)

# Patch grow.py to disable MERGE (Action Index 3)
grow_path = 'soma_research/soma/core/grow.py'
with open(grow_path, 'r') as f: grow_content = f.read()
# Force MERGE probability to 0 in select_action
grow_content = grow_content.replace("probs = torch.softmax(logits, dim=-1)", "logits.data[3] = -float('inf')\n        probs = torch.softmax(logits, dim=-1)")
with open(grow_path, 'w') as f: f.write(grow_content)

# 2. Reload modules
import soma.core.learn
import soma.core.grow
import soma.experiments.run_permuted_mnist
importlib.reload(soma.core.learn)
importlib.reload(soma.core.grow)
importlib.reload(soma.experiments.run_permuted_mnist)

# 3. Run fixed experiment
from soma.experiments.run_permuted_mnist import run_experiment
import argparse

args = argparse.Namespace(
    n_tasks=10, n_train=1000, n_test=200,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    seed=42, no_rl=False, disable_n1=False, disable_n2=False, disable_n3=False,
)

result = run_experiment(args)

=== SOMA Experiment 1: Permuted MNIST ===
Tasks: 10, Train: 1000, Test: 200
Device: cuda, Seed: 42

Generating permuted MNIST tasks...
[Task 1/10]
  Action: SPAWN(cold)  K: 0->1  Acc: 0.810  BT: 0.0000
[Task 2/10]
  Action: SPAWN(cold)  K: 1->2  Acc: 0.845  BT: 0.0000
[Task 3/10]
  Action: UPDATE_EXISTING  K: 2->2  Acc: 0.090  BT: 0.0000
[Task 4/10]
  Action: SPAWN_NEW  K: 2->3  Acc: 0.850  BT: 0.0000
[Task 5/10]
  Action: SPAWN_NEW  K: 3->4  Acc: 0.800  BT: 0.0000
[Task 6/10]
  Action: SKIP  K: 4->4  Acc: 0.205  BT: 0.0000
[Task 7/10]
  Action: SPAWN_NEW  K: 4->5  Acc: 0.825  BT: 0.0025
[Task 8/10]
  Action: MERGE  K: 5->4  Acc: 0.125  BT: -0.1521
[Task 9/10]
  Action: UPDATE_EXISTING  K: 4->4  Acc: 0.185  BT: -0.1331
[Task 10/10]
  Action: MERGE  K: 4->3  Acc: 0.210  BT: -0.2328

=== SOMA Summary ===
  backward_transfer        : -0.2328
  forward_transfer         : 0.0000
  final_k                  : 3
  spawn_count              : 5
  merge_count              : 2
  tasks_completed   

In [21]:
# Verify PASS/FAIL
bt = result['backward_transfer']
k = result['final_k']
passed = bt > -0.05 and k < 10
print(f'\nResult: {"PASS" if passed else "FAIL"}')
print(f'BT = {bt:.4f} (target > -0.05)')
print(f'K  = {k} (target < 10)')


Result: FAIL
BT = -0.2328 (target > -0.05)
K  = 3 (target < 10)
